# DAY1 실습 01-1 — 데이터 구조와 범주형 기술통계

In [ ]:
%pip install -q koreanize-matplotlib   # 한글 폰트 자동 설정 패키지. -q 는 설치 메시지 생략. Colab 은 세션마다 다시 설치한다

import numpy as np
import pandas as pd
import koreanize_matplotlib

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "ai4i2020.csv")

df = pd.read_csv(csv_path)

data = df.copy()

**원본과 사본:** `df`는 처음 읽은 원본을 보존하고 실습에는 `data = df.copy()`로 만든 사본을 사용한다. 사본에서 열을 추가해도 원본 표를 바꾸지 않는다.

AI4I 2020은 산업의 예측정비 상황을 모사해 만든 **합성 데이터**이다. 한 행은 하나의 제품 관측 레코드이다.

**토크 단위:** CSV의 `Torque [Nm]`는 토크이며 단위 Nm는 **뉴턴미터**로 읽는다. 분산의 단위는 (Nm)², 표준편차의 단위는 Nm이다. 코드의 실제 열 이름은 `Torque [Nm]` 그대로 사용한다.

## 기술통계란

### 기술통계의 목적

기술통계(Descriptive Statistics)는 관측한 데이터를 정리·요약·시각화하여 데이터의 **중심, 퍼짐, 모양과 이상값 후보**를 설명하는 통계 분석이다.

| 확인할 특징 | 답하는 질문 | 대표적인 방법 |
|---|---|---|
| 중심(Center) | 데이터가 어디에 모여 있는가? | 평균, 중앙값, 최빈값 |
| 퍼짐(Spread) | 데이터가 얼마나 흩어져 있는가? | 범위, IQR, 분산, 표준편차 |
| 모양(Shape) | 데이터 분포가 어떤 형태인가? | 대칭성, 치우침, 꼬리, 봉우리 |
| 이상값 후보 | 다른 관측값과 비교하여 매우 크거나 작은 값이 있는가? | 원자료, 박스플롯, IQR 기준 |


### 통계량을 해석할 때 단위도 함께 확인한다

기술통계량은 계산 방법에 따라 원자료의 단위를 그대로 가지기도 하고, 단위가 달라지거나 없어지기도 한다.

| 통계량 | 단위 | 해석할 때 확인할 점 |
|---|---|---|
| 평균 | 원자료와 같은 단위 | 값의 중심 수준을 원래 측정 단위로 해석한다 |
| 중앙값 | 원자료와 같은 단위 | 자료를 정렬했을 때 가운데에 위치하는 값이다 |
| 최소값·최대값 | 원자료와 같은 단위 | 실제 관측 범위를 확인한다 |
| 범위 | 원자료와 같은 단위 | `최대값 - 최소값`으로 전체 퍼짐을 나타낸다 |
| Q1·Q3 | 원자료와 같은 단위 | 자료의 25%, 75% 위치를 나타낸다 |
| IQR | 원자료와 같은 단위 | `Q3 - Q1`로 가운데 50%의 퍼짐을 나타낸다 |
| 표준편차 | 원자료와 같은 단위 | 값들이 평균 주변에서 얼마나 퍼져 있는지 나타낸다 |
| 분산 | 원자료 단위의 제곱 | 편차를 제곱하여 계산하므로 원자료와 단위가 다르다 |
| z-score | 단위 없음 | 평균에서 표준편차 몇 배만큼 떨어져 있는지 나타낸다 |

#### 분석할 때 함께 확인할 내용

- 전체 행 수와 실제 분석에 사용되는 **유효관측수**는 다를 수 있다.
- 실습 자료는 표본이므로 표준편차는 표본표준편차(`ddof=1`)로 계산한다.
- IQR이나 z-score 기준을 벗어난 값은 **이상값 후보**이며, 오류값이나 삭제 대상으로 바로 판단하지 않는다.
- 숫자로 저장된 변수라도 식별자라면 평균이나 표준편차를 계산할 분석변수로 사용하지 않는다.
- 계산 결과를 해석할 때는 가능한 경우 숫자와 함께 실제 측정 단위를 표시한다.

## 1. 데이터 구조 확인

행은 하나의 관측 기록, 열은 그 기록의 속성이다. `head()`는 앞부분만 보여 주므로 전체 자료의 품질을 대신 확인해 주지 않는다.

`head()`로 값의 모습을 보고, `shape`·`columns`·`info()`로 크기·열·저장 자료형을 확인한다. `isna()`는 결측 표시, `duplicated()`는 완전 중복 표시를 만들어 이후 요약의 기초를 점검한다.

공식 문서 : [head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html) · [shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html) · [info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) · [isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) · [duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)

In [ ]:
# [예제]
# 데이터의 처음 다섯 행에는 어떤 변수와 값이 기록되어 있는가?

# head : 앞에서부터 다섯 행을 보여 준다. 괄호 안에 숫자를 넣으면 그 개수만큼 보여 준다
data.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [ ]:
# [예제]
# 데이터는 몇 행 몇 열이며 어떤 변수를 포함하고 있는가?

# shape : (행 수, 열 수)  /  columns : 열 이름 목록. tolist 로 리스트 모양으로 바꿔 출력한다
print("행·열:", data.shape)
print("열 이름:", data.columns.tolist())

행·열: (10000, 14)
열 이름: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


In [ ]:
# [예제]
# 각 변수의 저장 자료형과 결측이 아닌 값의 수는 무엇인가?

# info : 열마다 결측이 아닌 값의 수(Non-Null Count)와 저장 자료형(Dtype)을 한 번에 보여 준다
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

### `isna()`의 의미

isna : 각 위치가 결측인지 확인한다. 결측은 True, 그 밖의 값은 False로 표시한다.

### `duplicated()`의 의미

duplicated : 기본적으로 모든 열의 값이 같은 중복 행을 확인한다. `keep="first"`가 기본값이므로 첫 번째 행은 `False`, 이후에 중복되는 행은 `True`로 표시한다.


In [ ]:
# [예제]
# 변수별 결측값은 몇 개이며 모든 열의 값이 같은 완전 중복 행은 몇 개인가?

# display : 표를 노트북의 표 모양으로 출력한다 (Colab 에서는 import 없이 쓸 수 있다)
# rename : 결과에 "결측값 수" 라는 이름을 붙인다
display(data.isna().sum().rename("결측값 수"))
print("완전 중복 행:", data.duplicated().sum())   # True 는 1, False 는 0 으로 계산되므로 sum() 이 중복 행 수가 된다

,결측값 수
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


완전 중복 행: 0


In [ ]:
# [실습]
# data의 처음 3행을 출력하고, Torque [Nm] 열의 결측값 수와 완전 중복 행 수를 확인하시오.


## 2. 통계적 유형과 분석 역할

| 변수 | 통계적 유형 | 분석 역할 | 해석 기준 |
|---|---|---|---|
| UDI, Product ID | 명목형 범주형 | 식별자 | 평균보다 고유값·중복 확인 |
| Type | 명목형 범주형 | 제품 유형 구분 (L·M·H) | 범주별 빈도·비율로 요약 |
| Machine failure | 이진 범주형 | 결과 상태 | 0=고장 없음, 1=고장 |
| Air temperature [K], Process temperature [K] | 수치형 | 온도 측정값 | 단위 K |
| Rotational speed [rpm] | 수치형 | 회전속도 측정값 | 정수로 기록된 rpm |
| Torque [Nm] | 수치형 | 토크 측정값 | 단위 Nm |
| Tool wear [min] | 수치형 | 누적 공구마모 시간 | 단위 min |
| TWF, HDF, PWF, OSF, RNF | 이진 범주형 | 고장 유형 표시 | 0=해당 없음, 1=해당 고장 |



## 3. 빈도·비율·최빈값

| 용어 | 의미 | 계산 |
|---|---|---|
| 빈도 | 해당 범주의 개수 | `value_counts()` |
| 비율 | 유효 관측수 중 해당 범주의 몫 | `value_counts(normalize=True)` |
| 백분율 | 비율을 100배 한 값 | 비율 × 100 |
| 최빈값 | 가장 자주 나온 값, 여러 개일 수 있음 | `mode()` |


공식 문서 : [value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [mode](https://pandas.pydata.org/docs/reference/api/pandas.Series.mode.html)

In [ ]:
# [예제]
# Type의 각 범주에는 관측값이 몇 개씩 있는가?

# value_counts : 범주마다 몇 번 나왔는지 센다. 많은 순서로 정렬해서 보여 준다
type_cnt = data["Type"].value_counts()
type_cnt

,count
Type,
L,6000
M,2997
H,1003


In [ ]:
# [예제] Type별 비율·백분율 표 만들기
# 1) Type별 비율과 백분율을 담은 데이터프레임 만들기 (소수점 4자리까지 반올림)
type_rate = data["Type"].value_counts(normalize=True)

# 계산한 비율과 백분율(비율 * 100)을 묶어서 데이터프레임으로 만들고, 소수점 4자리로 반올림한다.
df_rate = pd.DataFrame({"비율": type_rate, "백분율(%)": type_rate * 100}).round(4)

# 3) 행 이름 바꾸기 : rename(index={기존 이름: 새 이름}) 으로 L·M·H 를 알아보기 쉬운 이름으로 바꾼다
df_rate = df_rate.rename(index={"L": "저품질(L)", "M": "중품질(M)", "H": "고품질(H)"})


display(df_rate)

# 결측치를 제외하고 실제 비율 계산의 기준이 된 유효 데이터의 총 개수(분모)를 출력한다.
# count : 결측을 뺀 값의 개수
print("분모:", data["Type"].count())

            비율  백분율(%)
Type                  
저품질(L)  0.6000   60.00
중품질(M)  0.2997   29.97
고품질(H)  0.1003   10.03

분모: 10000


In [ ]:
# [예제]
# Type에서 가장 자주 나타나는 범주인 최빈값은 무엇인가?

# mode : 가장 자주 나온 값
data["Type"].mode()

,Type
0,L


In [ ]:
# [실습]
# Machine failure 열의 빈도와 최빈값을 구하시오.


### 결측값이 있을 때의 분모


같은 빈도라도 분모가 달라지면 비율이 달라진다.

- 전체 행 기준 : 결측을 포함한 전체 행 수(`len()`)로 나눈다.
- 유효값 기준 : 결측을 뺀 값의 수(`count()`)로 나눈다. `value_counts(normalize=True)`와 `mean()`은 유효값 기준으로 계산한다.

비율을 보고할 때는 어느 기준으로 계산했는지 함께 밝힌다.

In [ ]:
# [예제] 전체 행 수와 유효값 수에 따른 비율 비교
# 문제: 결측값(NaN)이 포함된 데이터에서 A의 비율을 계산한다.
#       전체 행 수와 유효값 수를 기준으로 계산한 결과를 비교한다.


# 1) 범주형 결측값이 포함된 Series 만들기---------------------------------
status = pd.Series(["A", "A", "B", np.nan], name="status")


# 2) A의 개수 계산---------------------------------
#    eq("A") : 값이 "A"이면 True, 아니면 False
#    sum()   : True를 1, False를 0으로 계산하여 A의 개수를 센다.
#    [True, True, False, False] → 2

cnt = status.eq("A").sum()
print("A 개수:", cnt)  # 2


# 3) 전체 행 수와 유효값 수 계산---------------------------------
#    len()   : 결측값(NaN)을 포함한 전체 데이터 개수
#    count() : 결측값(NaN)을 제외한 유효 데이터 개수

n_all = len(status)      # 4개 (A, A, B, NaN)
n_valid = status.count() # 3개 (A, A, B)

print("전체 행 수:", n_all)    # 4
print("유효값 수:", n_valid)   # 3


# 4) A의 비율 계산---------------------------------
#    전체 행 기준 : 2 / 4 = 0.5    → 50%
#    유효값 기준  : 2 / 3 = 0.6667 → 약 66.67%

comp = pd.DataFrame({
    "기준": ["전체 행", "유효값"],
    "분자_A빈도": [cnt, cnt],
    "분모": [n_all, n_valid],
    "A비율": [cnt / n_all, cnt / n_valid],
})

display(comp)

## 4. 고장 상태와 0/1 평균

Machine failure는 0=고장 없음, 1=고장이다. 0/1 변수는 합계가 1의 개수이므로 **평균=유효값 중 1의 비율**이다.

공식 문서 : [mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html) · [count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html)

In [ ]:
# [예제] 전체 건수와 유효 관측수에 따른 고장률 비교
# 0 = 고장 없음, 1 = 고장, None = 결측값 (0/1 이진 변수)

failure = pd.Series([0, 1, 0, None, 1])


# 1) 전체 건수와 유효 관측수 확인

print("전체 건수:", len(failure))       # 5
print("유효 관측수:", failure.count())  # 4


# 2) 고장 건수 확인
# 값이 1이면 True → True를 1로 계산하여 합산

fail_count = (failure == 1).sum()
print("고장 건수:", fail_count)         # 2


# 3) 전체 건수를 기준으로 고장률 계산
# 고장 2건 / 전체 5건 = 0.4

print("전체 기준 고장률:", fail_count / len(failure))
# 0.4


# 4) 유효 관측수를 기준으로 고장률 계산
# 고장 2건 / 유효 관측수 4건 = 0.5

print("유효값 기준 고장률:", fail_count / failure.count())
# 0.5


# 5) 0과 1로 된 데이터는 평균으로도 고장률 계산 가능
# mean() 은 결측값을 제외하고 계산한다.

print("평균으로 계산한 고장률:", failure.mean())
# 0.5

In [ ]:
# 6) value_counts()로 고장 여부별 빈도 계산
# 기본적으로 결측값은 제외한다.
# None 때문에 정수형이 실수형으로 변환되어 0.0, 1.0 으로 나온다
# sort_index : 많은 순서 대신 값(0, 1) 순서로 정렬한다
print("고장 여부별 건수:")
print(failure.value_counts().sort_index())
# 0.0    2
# 1.0    2

# 고장 여부별 건수와 결측값까지 확인
print("고장 여부별 건수:")
print(failure.value_counts(dropna=False).sort_index())

In [ ]:

failure = pd.Series([0, 1, 0, None, 1], dtype="Int64") #정수



print("유효값 기준 비율:")
print(failure.value_counts(normalize=True).sort_index())
# 0    0.5
# 1    0.5


print("전체 기준 비율:")
print(failure.value_counts(normalize=True, dropna=False))
# 0       0.4
# 1       0.4
# <NA>    0.2

In [ ]:
# [실습] 설비 고장 여부의 빈도와 비율 계산
# 문제: Machine failure 열에서 고장 없음(0)과 고장(1)의
#       건수, 비율, 백분율을 계산하여 표로 출력한다.
#       열 이름은 빈도, 비율, 백분율(%) 로 하고, 결과 표는 result 라는 이름에 담는다.
#       행 이름은 0 → 정상, 1 → 고장 으로 바꾼다


# 1) 고장 여부별 건수(빈도) 계산
#    value_counts() : 0과 1이 각각 몇 번 나오는지 계산한다.
#    sort_index()   : 인덱스를 0, 1 순서로 정렬한다.
#    0 = 고장 없음, 1 = 고장



# 2) 고장 여부별 비율 계산
#    normalize=True : 전체 유효 데이터에서 차지하는 비율을 구한다.
#    비율 = 해당 값의 건수 / 전체 유효 데이터 건수
#    예: 100건 중 고장 5건 → 5 / 100 = 0.05



# 3) 빈도, 비율, 백분율을 하나의 표로 만들기
#    빈도       : 실제 데이터 건수
#    비율       : 0~1 사이의 값
#    백분율(%)  : 비율 × 100



# 4) 행 이름 바꾸기 : rename(index=...) 로 0 → "정상", 1 → "고장"


### 직접 해보기 1 — HDF 빈도와 비율

`HDF`(열 방출 고장)의 `0`(고장 없음), `1`(고장 발생)에 대해 빈도와 비율을 한 표로 계산한다.

1. `data["HDF"]`의 빈도는 `value_counts()`로 계산한다.
2. 비율은 `value_counts(normalize=True)`로 계산한다.
3. `빈도`, `비율` 두 열로 DataFrame을 만들고 0과 1 순서로 출력한다.

**확인 :** `0`은 9,885건, `1`은 115건이다.


In [ ]:
# 직접 해보기 1 — HDF 빈도와 비율



### 직접 해보기 2 — TWF 고장 건수와 백분율

`TWF`(공구 마모 고장)에서 고장 발생(`1`)의 건수와 백분율을 구한다.

1. `data["TWF"]`에서 `1`인 값의 개수를 센다.
2. 0/1 변수의 평균에 100을 곱하여 고장 백분율(%)을 구한다.
3. 백분율은 소수 둘째 자리까지 출력한다.

**실행 결과**

```text
고장 건수: 46
고장 백분율(%): 0.46
```


In [ ]:
# 직접 해보기 2 — TWF 고장 건수와 백분율



## 5. 정리

| 확인할 것 | 함수 | 답하는 질문 |
|---|---|---|
| 데이터 | `head()` | 어떤 값이 기록되어 있나 |
| 크기·열 이름 | `shape`, `columns` | 몇 행 몇 열, 어떤 변수가 있나 |
| 자료형·결측 아닌 수 | `info()` | 어떻게 저장되었나, 빠진 값이 있나 |
| 결측 개수 | `isna().sum()` | 열마다 결측이 몇 개인가 |
| 완전 중복 행 수 | `duplicated().sum()` | 같은 행이 몇 번 더 들어갔나 |
| 빈도 | `value_counts()` | 범주마다 몇 개인가 |
| 비율 | `value_counts(normalize=True)` | 범주마다 몇 %인가 (유효값 기준) |
| 결측 포함 비율 | `value_counts(normalize=True, dropna=False)` | 전체 행 기준으로 몇 %인가 |
| 최빈값 | `mode()` | 가장 흔한 범주는 무엇인가 (여러 개일 수 있다) |
| 0/1 비율 | `mean()` | 1(고장)의 비율은 얼마인가 |

순서
1) 구조를 본다 (head → shape → info)
2) 결측과 중복을 확인한다 (isna → duplicated)
3) 범주형은 빈도·비율·최빈값으로, 0/1 변수는 평균으로 요약한다
4) 비율을 보고할 때는 분모(전체 행 기준인지, 유효값 기준인지)를 같이 적는다

공식 문서 : [value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [mode](https://pandas.pydata.org/docs/reference/api/pandas.Series.mode.html) · [isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) · [duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) · [mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html)

## 6. 종합 실습 : 결측과 중복이 있는 점검 기록

| 열 | 뜻 |
|---|---|
| 설비ID | 설비 번호 |
| 라인 | 생산 라인 (A·B·C) |
| 교대 | 주간·야간 |
| 고장 | 0 = 정상, 1 = 고장 |

### 중복 레코드 처리 절차

중복 레코드 : 모든 열의 값이 같은 행이 두 번 이상 기록된 것. 그대로 두면 같은 기록이 여러 번 집계되어 빈도와 비율이 실제와 달라진다. 따라서 빈도·비율을 계산하기 전에 아래 순서로 처리한다.

1) 탐지 : `duplicated().sum()`으로 중복 건수를 확인하고, `duplicated(keep=False)`로 중복된 행을 모두 출력해 내용을 직접 본다.
2) 원인 판단 : 같은 기록이 두 번 입력된 오류인지, 실제로 같은 값이 반복 측정된 정상 기록인지 구분한다. 설비ID·점검 시각처럼 기록 하나를 구별하는 열(식별 키)이 같으면 입력 오류로 보고, 식별 키가 다르면 정상 기록으로 보고 남긴다.
3) 제거 : `drop_duplicates()`로 중복 행 중 처음 나온 행만 남긴다. 원본은 바뀌지 않으므로 결과를 새 변수에 저장한다.
4) 검증 : 제거 전후의 행 수 차이가 1)에서 센 중복 건수와 같은지, 제거 후 `duplicated().sum()`이 0인지 확인한다.
5) 집계 : 검증을 마친 데이터로 빈도·비율을 계산한다. 보고할 때는 제거한 중복 건수를 함께 적는다.

공식 문서 : [drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) · [duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)

In [ ]:
# [데이터 준비]
records = pd.DataFrame({
    "설비ID": ["E01", "E02", "E03", "E04", "E05", "E06", "E07", "E08", "E09", "E06"],
    "라인":   ["A",   "A",   "B",   "B",   "C",   "A",   "B",   "C",   np.nan, "A"],
    "교대":   ["주간", "야간", "주간", "야간", "주간", "주간", "야간", "야간", "주간", "주간"],
    "고장":   [0,     1,     0,     np.nan, 1,    0,     0,     np.nan, 1,     0],
})
records

In [ ]:
# 문제 1)
# 몇 행 몇 열이고, 각 열의 저장 자료형은 무엇인가?



In [ ]:
# 문제 2)
# 열별 결측값 수와 완전 중복 행 수는 몇 개인가?



In [ ]:
# 문제 3)
# 중복된 행을 직접 출력해 보고, 중복을 제거한 뒤 행 수와 중복 수를 다시 확인하시오.
# 중복을 제거한 표는 check 라는 이름으로 저장한다.
# 힌트 : duplicated(keep=False) 는 처음 나온 행까지 포함해 중복된 행을 모두 True 로 표시한다.



### 종합 실습 4~6 데이터 준비


In [ ]:
# [데이터 준비] 문제 3을 풀지 못한 경우 이 셀을 실행한 뒤 문제 4~6을 푼다
# drop_duplicates() : 중복 행 가운데 처음 나온 행만 남긴 새 표를 돌려준다 (records 는 그대로)
check = records.drop_duplicates()

In [ ]:
# 문제 4)
# 라인별 빈도와 비율(유효값 기준)을 중복 제거 전(records)과 후(check)로 나란히 놓은 표를 만들고 비교하시오.



In [ ]:
# 문제 5)
# 중복을 제거한 표(check)에서 라인의 최빈값은 무엇인가? 결과가 몇 개 나오며 왜 그런가?



In [ ]:
# 문제 6)
# 중복을 제거한 표(check)에서 고장률을 전체 행 기준과 유효값 기준으로 각각 구하시오


